# Chapter 14: Boltzmann Machines


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Every model so far has been *discriminative*.  Given an input we predicted
an output: a number, a class, a solution to a differential equation, a
reconstruction.  Even the autoencoder of Chapter 12, which had no
labels, was scored on how well it reproduced a given input.

A *generative* model asks a different question.  Rather than
$p(y\mid\bm{x})$ it models $p(\bm{x})$ itself: not what to predict from an
observation, but what observations are likely at all.  With such a model in hand
one can sample new data, evaluate how surprising an observation is, fill in
missing components, and -- most valuable in the physical sciences -- inspect the
structure the model has inferred.

The Boltzmann machine is the oldest such model in this book and the one closest
to physics, because it *is* physics: the distribution it fits is the
canonical ensemble, its parameters are couplings and fields, and the difficulty
of training it is the difficulty of computing a partition function.  That
difficulty is genuine and is the organising theme of the chapter.  We derive it,
prove exactly where it enters, and then spend the rest of the chapter on the
Markov chain Monte Carlo machinery invented to get around it.

The material follows the lecture notes for weeks eleven and twelve of
FYS-STK3155/4155 and the accompanying notes on Gibbs sampling.


## Energy models

Let $\bm{x}\in\mathcal{X}$ be a configuration of the variables we observe -- the
*visible* units -- and let $\bm{h}\in\mathcal{H}$ be a configuration of
variables we do not observe, the *hidden* units.  An energy-based model
assigns every joint configuration an energy $E(\bm{x},\bm{h};\bm{\Theta})$ and
declares the probability to be the Boltzmann weight of that energy,

$$
\boxed{\;
  p(\bm{x},\bm{h};\bm{\Theta})
  = \frac{f(\bm{x},\bm{h};\bm{\Theta})}{Z(\bm{\Theta})},
  \qquad
  f = e^{-E(\bm{x},\bm{h};\bm{\Theta})},\;}\tag{14.1}
$$

normalised by the *partition function*

$$
Z(\bm{\Theta}) = \sum_{\bm{x}}\sum_{\bm{h}}
    f(\bm{x},\bm{h};\bm{\Theta}).\tag{14.2}
$$

The parameters $\bm{\Theta}$ are what we fit.  Anything non-negative can be
written as $e^{-E}$, so Eq. (14.1) is not a restriction on the
family of distributions; it is a change of variables that makes the structure
convenient, and it is the same change of variables a physicist makes when
writing $p\propto e^{-\beta H}$.

Two remarks on notation.  A symbol like $\bm{x}$ now denotes an entire
*configuration*, not a single variable, and the sums in
Eq. (14.2) run over all configurations.  And the temperature has been
absorbed into $E$; setting $\beta=1$ costs nothing because $\bm{\Theta}$ can
rescale.

Since the hidden units are not observed, what we can compare against data is the
marginal

$$
p(\bm{x};\bm{\Theta})
  = \sum_{\bm{h}} p(\bm{x},\bm{h};\bm{\Theta})
  = \frac{1}{Z(\bm{\Theta})}\sum_{\bm{h}} f(\bm{x},\bm{h};\bm{\Theta})
  \;=:\; \frac{f(\bm{x};\bm{\Theta})}{Z(\bm{\Theta})},\tag{14.3}
$$

and it is convenient to name the exponent of the marginal.

```{admonition} Definition (Free energy)
:class: important
The *free energy* of a visible configuration is

$$
F(\bm{x};\bm{\Theta})
   = -\log\sum_{\bm{h}}e^{-E(\bm{x},\bm{h};\bm{\Theta})},
  \qquad\text{so that}\qquad
  p(\bm{x};\bm{\Theta}) = \frac{e^{-F(\bm{x};\bm{\Theta})}}{Z(\bm{\Theta})}.\tag{14.4}
$$
```

The name is the physicist's: $F$ is the free energy of the hidden subsystem at
fixed visible configuration, and the sum over $\bm{h}$ is a partial trace.  For
the restricted machines of Section *The restriction* that sum can be done in closed
form, which is the whole reason they are used.

### The counting problem

Equation (14.2) is a sum over every configuration.  If there are $M$
binary visible units and $N$ binary hidden units it has $2^{M}\times2^{N}$
terms.  For a machine with $M=784$ visible units, one per pixel of an MNIST
image, and $N=500$ hidden units, that is $2^{1284}\approx10^{386}$ terms -- more
than the number of atoms in the observable universe raised to a considerable
power.  The partition function is not merely expensive; it is permanently out of
reach.

This is not a defect of the model.  It is the same intractability that makes
statistical mechanics hard, and the same one that made the Ising model resist
solution in three dimensions.  Everything technical in this chapter is a
response to it.


## Maximum likelihood and the two phases

Given data $\bm{X}=\{\bm{x}^{(1)},\dots,\bm{x}^{(n)}\}$ assumed independent, the
likelihood is $\prod_v p(\bm{x}^{(v)};\bm{\Theta})$, and as always we work with
its logarithm,

$$
\mathcal{L}(\bm{\Theta})
   = \frac{1}{n}\sum_{v=1}^{n}\log p(\bm{x}^{(v)};\bm{\Theta})
   = -\frac{1}{n}\sum_{v=1}^{n}F(\bm{x}^{(v)};\bm{\Theta})
     - \log Z(\bm{\Theta}).\tag{14.5}
$$

The two terms behave very differently, and the following theorem says exactly
how.

```{admonition} Theorem (The gradient is a difference of two expectations)
:class: important
For the model (14.1),

$$
\boxed{\;
  \nabla_{\bm{\Theta}}\mathcal{L}
  = \underbrace{-\left\langle
      \nabla_{\bm{\Theta}}E(\bm{x},\bm{h})\right\rangle_{\mathrm{data}}}
      _{\text{positive phase}}
    \;+\;
    \underbrace{\left\langle
      \nabla_{\bm{\Theta}}E(\bm{x},\bm{h})\right\rangle_{\mathrm{model}}}
      _{\text{negative phase}},\;}\tag{14.6}
$$

where $\langle\cdot\rangle_{\mathrm{data}}$ averages over
$\bm{x}\sim\mathrm{data}$ and $\bm{h}\sim p(\bm{h}\mid\bm{x})$, and
$\langle\cdot\rangle_{\mathrm{model}}$ averages over the joint model
distribution $p(\bm{x},\bm{h};\bm{\Theta})$.
```

```{admonition} Proof
:class: note
Take the second term of Eq. (14.5) first.  Since
$\nabla\log Z=\nabla Z/Z$ and $Z=\sum_{\bm{x},\bm{h}}e^{-E}$,

$$
\nabla_{\bm{\Theta}}\log Z
  = \frac{1}{Z}\sum_{\bm{x},\bm{h}}
      \nabla_{\bm{\Theta}}e^{-E(\bm{x},\bm{h})}
  = -\frac{1}{Z}\sum_{\bm{x},\bm{h}}
      e^{-E(\bm{x},\bm{h})}\nabla_{\bm{\Theta}}E(\bm{x},\bm{h})
  = -\left\langle\nabla_{\bm{\Theta}}E\right\rangle_{\mathrm{model}},
$$

recognising $e^{-E}/Z$ as $p(\bm{x},\bm{h})$.  For the first term, by
Definition def:14-freeenergy,

$$
-\nabla_{\bm{\Theta}}F(\bm{x})
  = \nabla_{\bm{\Theta}}\log\sum_{\bm{h}}e^{-E(\bm{x},\bm{h})}
  = -\frac{\sum_{\bm{h}}e^{-E}\nabla_{\bm{\Theta}}E}
          {\sum_{\bm{h}}e^{-E}}
  = -\left\langle\nabla_{\bm{\Theta}}E\right\rangle_{p(\bm{h}\mid\bm{x})},
$$

since $e^{-E(\bm{x},\bm{h})}/\sum_{\bm{h}'}e^{-E(\bm{x},\bm{h}')}
=p(\bm{h}\mid\bm{x})$.  Averaging over the data and subtracting
$\nabla\log Z$ gives Eq. (14.6).
```

Equation (14.6) is the central formula of the chapter and it has
a vivid reading.  The positive phase lowers the energy of configurations the
data actually contain; the negative phase raises the energy of configurations
the *model* currently believes in.  Learning is a competition between the
two, and it stops exactly when the model's expectations match the data's --
which is the classical method-of-moments condition of
Chapter 2, arrived at from a different direction.

```{admonition} Where the difficulty lives
:class: tip
The positive phase is an average over the
training set and costs nothing.  The negative phase is an average over the
model distribution, which is $e^{-E}/Z$, and *drawing samples from it is
exactly the problem that $Z$ makes hard*.  Every technique in the rest of this
chapter -- Markov chains, Metropolis, Gibbs, contrastive divergence -- exists to
estimate that one expectation.  Note also what is *not* needed: nowhere does
Eq. (14.6) require the value of $Z$, only samples from the
distribution it normalises.  That is the loophole the whole subject exploits.
```

### The same statement as a divergence

There is an equivalent formulation worth having.  For distributions $p$ and $q$
the Kullback-Leibler divergence is

$$
\mathrm{KL}(p\,\|\,q) = \sum_{\bm{x}} p(\bm{x})\log\frac{p(\bm{x})}{q(\bm{x})}
   \;\ge\;0,\tag{14.7}
$$

with equality if and only if $p=q$, by Jensen's inequality applied to the convex
function $-\log$.  Writing $p_{\mathrm{data}}$ for the empirical distribution,

$$
\mathrm{KL}\!\left(p_{\mathrm{data}}\,\|\,p_{\bm{\Theta}}\right)
  = -H(p_{\mathrm{data}}) - \mathcal{L}(\bm{\Theta}),\tag{14.8}
$$

and since the entropy $H(p_{\mathrm{data}})$ does not depend on
$\bm{\Theta}$, *maximising the likelihood is minimising the KL divergence
from the data to the model*.  The two views are the same computation; the
divergence view makes the target explicit and will reappear when variational
methods are introduced.


## Markov chain Monte Carlo

We need samples from $p(\bm{x})\propto e^{-F(\bm{x})}$ without knowing the
constant.  The device is to construct a Markov chain whose long-run
distribution is the one we want, run it, and use its states as samples.

A Markov chain on a finite state space is a sequence $\bm{s}^{(0)},
\bm{s}^{(1)},\dots$ in which the next state depends only on the current one,

$$
\Prob\!\left(\bm{s}^{(t+1)}=\bm{y}\mid \bm{s}^{(t)}=\bm{x},
    \bm{s}^{(t-1)},\dots\right) = T(\bm{x}\to\bm{y}),\tag{14.9}
$$

with $T$ the transition matrix, $T(\bm{x}\to\bm{y})\ge0$ and
$\sum_{\bm{y}}T(\bm{x}\to\bm{y})=1$.  A distribution $\pi$ is *stationary*
for $T$ if it reproduces itself,

$$
\pi(\bm{y}) = \sum_{\bm{x}}\pi(\bm{x})T(\bm{x}\to\bm{y}).\tag{14.10}
$$

Two properties make a chain useful.  It is *irreducible* if every state can
be reached from every other, and *aperiodic* if it does not return to
states only at multiples of some period; together these are what is usually
called *ergodicity*, and the standard theorem then says that the chain has
a unique stationary distribution and converges to it from any start.  What
remains is to arrange for $\pi$ to be the distribution we want, and the
following condition is the standard tool.

```{admonition} Definition (Detailed balance)
:class: important
A transition matrix $T$ satisfies *detailed balance* with respect to $\pi$
if

$$
\pi(\bm{x})\,T(\bm{x}\to\bm{y}) = \pi(\bm{y})\,T(\bm{y}\to\bm{x})
  \qquad\text{for all }\bm{x},\bm{y}.\tag{14.11}
$$
```

```{admonition} Proposition (Detailed balance implies stationarity)
:class: important
If $T$ satisfies Eq. (14.11) then $\pi$ is stationary for $T$.
The converse is false.
```

```{admonition} Proof
:class: note
Summing Eq. (14.11) over $\bm{x}$,

$$
\sum_{\bm{x}}\pi(\bm{x})T(\bm{x}\to\bm{y})
  = \sum_{\bm{x}}\pi(\bm{y})T(\bm{y}\to\bm{x})
  = \pi(\bm{y})\sum_{\bm{x}}T(\bm{y}\to\bm{x})
  = \pi(\bm{y}),
$$

using that the rows of $T$ sum to one.  This is
Eq. (14.10).  For the converse, a deterministic cycle on three
states has a uniform stationary distribution but no reverse transitions at all,
so Eq. (14.11) fails.
```

Detailed balance is a physical statement: in equilibrium the flow of probability
from $\bm{x}$ to $\bm{y}$ exactly cancels the flow back.  It is stronger than
stationarity but far easier to check, and every algorithm below is designed
around it.

### Metropolis-Hastings

Propose a move from $\bm{x}$ to $\bm{y}$ with some proposal density
$q(\bm{x}\to\bm{y})$, and accept it with probability

$$
A(\bm{x}\to\bm{y}) = \min\!\left\{1,\;
    \frac{\pi(\bm{y})\,q(\bm{y}\to\bm{x})}
         {\pi(\bm{x})\,q(\bm{x}\to\bm{y})}\right\}.\tag{14.12}
$$

If rejected, the chain stays where it is.

```{admonition} Proposition
:class: important
The chain defined by Eq. (14.12) satisfies detailed balance
with respect to $\pi$, and requires $\pi$ only through ratios
$\pi(\bm{y})/\pi(\bm{x})$.
```

```{admonition} Proof
:class: note
The full transition for $\bm{y}\neq\bm{x}$ is $T=qA$.  Suppose without loss of
generality that the ratio in Eq. (14.12) is at most one, so
$A(\bm{x}\to\bm{y})=\pi(\bm{y})q(\bm{y}\to\bm{x})/
[\pi(\bm{x})q(\bm{x}\to\bm{y})]$ and $A(\bm{y}\to\bm{x})=1$.  Then

$$
\pi(\bm{x})q(\bm{x}\to\bm{y})A(\bm{x}\to\bm{y})
  = \pi(\bm{y})q(\bm{y}\to\bm{x})
  = \pi(\bm{y})q(\bm{y}\to\bm{x})A(\bm{y}\to\bm{x}),
$$

which is Eq. (14.11).  The case $\bm{x}=\bm{y}$ is trivial.  The
ratio statement is immediate: $\pi$ enters only as $\pi(\bm{y})/\pi(\bm{x})
=e^{-[F(\bm{y})-F(\bm{x})]}$, in which $Z$ cancels.
```

That cancellation is the point.  Proposition prop:14-metropolis is what
makes sampling from $e^{-F}/Z$ possible without ever computing $Z$, and it is
the loophole promised in the notebox of Section *Maximum likelihood and the two phases*.

### Gibbs sampling

Gibbs sampling replaces the proposal-and-accept step by a sequence of exact
conditional draws.  With the state split into components
$\bm{s}=(s_1,\dots,s_d)$, one sweep visits each component and replaces it by a
draw from its conditional given all the others,

$$
s_i \;\sim\; \pi\!\left(s_i \mid s_1,\dots,s_{i-1},s_{i+1},\dots,s_d\right).\tag{14.13}
$$

```{admonition} Proposition (Gibbs is Metropolis with acceptance one)
:class: important
The update (14.13) satisfies detailed balance with respect to $\pi$,
and is the special case of Eq. (14.12) in which the proposal is
the exact conditional and every proposal is accepted.
```

```{admonition} Proof
:class: note
Write $\bm{s}_{-i}$ for the components other than $i$, and take
$q(\bm{x}\to\bm{y})=\pi(y_i\mid\bm{x}_{-i})$ for $\bm{y}$ differing from
$\bm{x}$ only in component $i$.  Then, using
$\pi(\bm{x})=\pi(x_i\mid\bm{x}_{-i})\pi(\bm{x}_{-i})$ and
$\bm{x}_{-i}=\bm{y}_{-i}$,

$$
\frac{\pi(\bm{y})q(\bm{y}\to\bm{x})}{\pi(\bm{x})q(\bm{x}\to\bm{y})}
  = \frac{\pi(y_i\mid\bm{x}_{-i})\pi(\bm{x}_{-i})\,\pi(x_i\mid\bm{x}_{-i})}
         {\pi(x_i\mid\bm{x}_{-i})\pi(\bm{x}_{-i})\,\pi(y_i\mid\bm{x}_{-i})}
  = 1,
$$

so the acceptance probability of Eq. (14.12) is $\min\{1,1\}=1$
and detailed balance holds by Proposition prop:14-metropolis.
```

So Gibbs sampling never rejects, which is its great practical advantage -- no
tuning of a step size, no wasted proposals.  Its cost is that it needs the
conditionals in closed form and it moves one component at a time, so strongly
correlated components make it mix slowly.  Metropolis, by contrast, works with
any proposal but throws work away.  Which is better depends entirely on whether
the conditionals are available, and for the restricted machines of the next
section they are available and are trivially cheap.


## Boltzmann machines

A *Boltzmann machine* is the energy model (14.1) with a
quadratic energy on binary units.  Writing $\bm{s}=(\bm{x},\bm{h})$ for all
units together,

$$
E(\bm{s}) = -\sum_{i} c_i s_i - \sum_{i<j} J_{ij}s_is_j,\tag{14.14}
$$

which any physicist will recognise as an Ising model with fields $c_i$ and
couplings $J_{ij}$, and which any statistician will recognise as a pairwise
undirected graphical model.  The visible units are clamped to data during the
positive phase; the hidden units are never observed and exist to let the
marginal $p(\bm{x})$ be richer than the pairwise form would otherwise allow.

The general machine of Eq. (14.14) is essentially untrainable.
Every unit couples to every other, so no conditional factorises, Gibbs sampling
must be done one unit at a time, and *both* phases of
Eq. (14.6) require Monte Carlo -- the positive phase too,
because with visible-visible and hidden-hidden couplings even
$p(\bm{h}\mid\bm{x})$ is intractable.  Two nested approximations, each noisy,
compound into an algorithm that in practice does not converge.

### The restriction

The remedy is to forbid connections within a layer.  A *restricted*
Boltzmann machine keeps only visible-hidden couplings, so its graph is
bipartite:

$$
E(\bm{x},\bm{h};\bm{\Theta})
   = -\bm{a}^{\mathsf{T}}\bm{x} - \bm{b}^{\mathsf{T}}\bm{h}
     - \bm{x}^{\mathsf{T}}\bm{W}\bm{h},\tag{14.15}
$$

with visible biases $\bm{a}\in\mathbb{R}^{M}$, hidden biases
$\bm{b}\in\mathbb{R}^{N}$ and weights $\bm{W}\in\mathbb{R}^{M\times N}$.  This
single deletion changes everything, and the following theorem says why.

```{admonition} Theorem (Conditional independence)
:class: important
For the restricted machine (14.15) with binary units in
$\{0,1\}$, the conditionals factorise completely,

$$
p(\bm{h}\mid\bm{x}) = \prod_{j=1}^{N}p(h_j\mid\bm{x}),
  \qquad
  p(\bm{x}\mid\bm{h}) = \prod_{i=1}^{M}p(x_i\mid\bm{h}),\tag{14.16}
$$

with logistic on-probabilities

$$
p(h_j=1\mid\bm{x}) = \sigma\!\left(b_j + \bm{x}^{\mathsf{T}}\bm{w}_{\ast j}\right),
  \qquad
  p(x_i=1\mid\bm{h}) = \sigma\!\left(a_i + \bm{w}_{i\ast}^{\mathsf{T}}\bm{h}\right),\tag{14.17}
$$

where $\sigma(z)=1/(1+e^{-z})$.  Furthermore the free energy of
Definition def:14-freeenergy is available in closed form,

$$
F(\bm{x}) = -\bm{a}^{\mathsf{T}}\bm{x}
    - \sum_{j=1}^{N}\log\!\left(1+e^{\,b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j}}\right).\tag{14.18}
$$
```

```{admonition} Proof
:class: note
Sum the joint weight over the hidden configurations.  Because
Eq. (14.15) contains no $h_jh_{j'}$ term, the exponent is a sum
of terms each involving a single $h_j$, so the exponential factorises and the
sum over $\bm{h}\in\{0,1\}^{N}$ becomes a product of independent sums:

$$
\sum_{\bm{h}}e^{-E}
  = e^{\bm{a}^{\mathsf{T}}\bm{x}}\sum_{\bm{h}}
      \prod_{j}e^{(b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j})h_j}
  = e^{\bm{a}^{\mathsf{T}}\bm{x}}\prod_{j}
      \sum_{h_j\in\{0,1\}}e^{(b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j})h_j}
  = e^{\bm{a}^{\mathsf{T}}\bm{x}}\prod_{j}
      \left(1+e^{\,b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j}}\right).
$$

Taking $-\log$ gives Eq. (14.18).  Dividing the joint by this
marginal, the factor $e^{\bm{a}^{\mathsf{T}}\bm{x}}$ cancels and

$$
p(\bm{h}\mid\bm{x})
  = \prod_{j}\frac{e^{(b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j})h_j}}
                  {1+e^{\,b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j}}},
$$

which is Eq. (14.16); setting $h_j=1$ and dividing numerator and
denominator by $e^{b_j+\bm{x}^{\mathsf{T}}\bm{w}_{\ast j}}$ gives the logistic
form.  The visible case is identical by the symmetry of
Eq. (14.15) under $\bm{x}\leftrightarrow\bm{h}$,
$\bm{a}\leftrightarrow\bm{b}$, $\bm{W}\leftrightarrow\bm{W}^{\mathsf{T}}$.
```

Theorem thm:14-factorise is what makes restricted machines usable, and it
buys three things at once.  The positive phase of
Eq. (14.6) becomes exact, since $p(\bm{h}\mid\bm{x})$ is known
in closed form and no sampling is needed.  Gibbs sampling can update an entire
layer at once -- *block* Gibbs --

$$
\bm{h}^{(t)}\sim p(\bm{h}\mid\bm{x}^{(t)}),
  \qquad
  \bm{x}^{(t+1)}\sim p(\bm{x}\mid\bm{h}^{(t)}),\tag{14.19}
$$

because within a layer the units are independent given the other layer, so
$M+N$ scalar updates collapse into two vector operations.  And the free
energy (14.18) is computable, so unnormalised probabilities can be
compared even though $Z$ cannot.

Note also what Eq. (14.17) says about the arithmetic: it is
$\sigma(\bm{W}^{\mathsf{T}}\bm{x}+\bm{b})$, which is precisely the forward pass
of a single logistic layer from Chapter 5.  An RBM is a
neural-network layer read as a probability model, with the same weights used in
both directions.

Specialising Eq. (14.6) to Eq. (14.15), where
$\partial E/\partial w_{ij}=-x_ih_j$, gives the form actually implemented:

$$
\frac{\partial\mathcal{L}}{\partial w_{ij}}
   = \left\langle x_ih_j\right\rangle_{\mathrm{data}}
   - \left\langle x_ih_j\right\rangle_{\mathrm{model}},
  \quad
  \frac{\partial\mathcal{L}}{\partial a_i}
   = \left\langle x_i\right\rangle_{\mathrm{data}}
   - \left\langle x_i\right\rangle_{\mathrm{model}},\tag{14.20}
$$

and similarly for $b_j$.  Every term is a correlation between a visible and a
hidden unit: learning matches the model's correlations to the data's.

### Gaussian-binary machines

Binary visible units are wrong for continuous data.  The Gaussian-binary machine
replaces them by real variables with a quadratic term,

$$
E_{GB}(\bm{x},\bm{h};\bm{\Theta})
   = \sum_{i=1}^{M}\frac{(x_i-a_i)^{2}}{2\sigma_i^{2}}
   - \sum_{j=1}^{N}b_jh_j
   - \sum_{i,j}\frac{x_i w_{ij} h_j}{\sigma_i^{2}},\tag{14.21}
$$

which leaves the hidden conditional unchanged in form,
$p(h_j=1\mid\bm{x})=\sigma(b_j+\sum_i x_iw_{ij}/\sigma_i^{2})$, and makes the
visible conditional a Gaussian,

$$
p(x_i\mid\bm{h}) = \mathcal{N}\!\left(a_i
    + \bm{w}_{i\ast}^{\mathsf{T}}\bm{h},\;\sigma_i^{2}\right).\tag{14.22}
$$

Block Gibbs works exactly as before, one layer at a time.  The variances
$\sigma_i^{2}$ are usually fixed to one after standardising the data, because
learning them jointly with $\bm{W}$ is notoriously unstable -- the energy can be
driven down by shrinking $\sigma_i$ without improving the fit.


## Contrastive divergence

Theorem thm:14-gradient left one term requiring samples from the model.
The honest procedure is to run the chain (14.19) to equilibrium
from any start, which takes an unknown and possibly enormous number of sweeps.
*Contrastive divergence* does something cheaper and admits it: start the
chain *at the data* and run it for $k$ sweeps only, usually $k=1$.

$$
\nabla_{\bm{\Theta}}\mathcal{L}
  \;\approx\;
  -\left\langle\nabla E\right\rangle_{\mathrm{data}}
  + \left\langle\nabla E\right\rangle_{k\text{ sweeps from the data}}.\tag{14.23}
$$

The reasoning is that the data are, if the model is any good, already close to
the model distribution, so a short chain moves in the direction the model is
wrong and that direction is what the gradient needs.

This is an approximation of a particular and awkward kind.

```{admonition} CD-$k$ is biased, not merely noisy
:class: tip
A noisy estimator has the right
expectation and averages out over updates; a biased one does not.  CD-$k$ is
biased for every finite $k$, and it is not the gradient of any function -- there
is no "contrastive divergence objective" that it descends.  What can be said
is that the bias vanishes as $k\to\infty$ and that in practice it points close
enough to the truth for gradient ascent to work.  Section *How biased is CD-\texorpdfstring{$k$}{k}?*
measures both halves of that claim.
```

A cheap improvement is *persistent* contrastive divergence, which keeps the
Gibbs chain running across parameter updates instead of restarting it at the
data each time.  Since the parameters move slowly, the chain stays near
equilibrium and the bias falls, at no extra cost per update.


## Implementation and verification

Everything in Sections *The restriction* and *Contrastive divergence* is a few lines, and small
machines let us check them against exact answers -- because for $M$ up to about
twenty, $Z$ *can* be enumerated.  That is the strategy of this section: use
brute force where it is available to audit the approximations we are forced to
use elsewhere.


In [ ]:
def free_energy(P, X):
    """F(x) = -a.x - sum_j log(1 + exp(b_j + (x^T W)_j)),  Eq. (14.freeenergy).

    The hidden units have been summed out exactly, which is what the
    restriction to a bipartite graph buys.
    """
    z = X @ P["W"] + P["b"]
    return -(X @ P["a"]) - np.sum(np.logaddexp(0.0, z), axis=-1)


def p_h_given_x(P, X):
    return sigmoid(X @ P["W"] + P["b"])


def p_x_given_h(P, H):
    return sigmoid(H @ P["W"].T + P["a"])


def gibbs_step(P, X, rng):
    """One sweep of block Gibbs: sample all h, then all x, Eq. (14.blockgibbs)."""
    ph = p_h_given_x(P, X)
    H = (rng.random(ph.shape) < ph).astype(float)
    px = p_x_given_h(P, H)
    Xn = (rng.random(px.shape) < px).astype(float)
    return Xn, H, ph, px


The exact gradient enumerates the $2^{M}$ visible states and weights them by the
true model probability; the CD-$k$ gradient replaces that sum by a short chain.


In [ ]:
def exact_gradient(P, X):
    """The true gradient of the log-likelihood, Eq. (14.gradient).

    The negative phase is computed by enumerating all 2^M visible states and
    weighting by the exact model distribution.  Only feasible for small M, and
    that is exactly why it is useful: it is the ground truth against which
    contrastive divergence can be judged.
    """
    pos = positive_phase(P, X)
    V = all_states(len(P["a"]))
    logp = -free_energy(P, V)
    logp = logp - np.logaddexp.reduce(logp)
    w = np.exp(logp)                                  # exact p(v)
    ph = p_h_given_x(P, V)
    neg = {"W": (V * w[:, None]).T @ ph, "a": w @ V, "b": w @ ph}
    return {k: pos[k] - neg[k] for k in pos}


**The gradient formula.** 
Theorem thm:14-gradient is an identity and can be checked as one, by
comparing \verb!exact_gradient! against central differences of the exactly
computed log-likelihood:


```
--- exact gradient vs finite differences of the log-likelihood ---
  W: max relative error over all 24 entries = 2.61e-08
  a: max relative error over all 6 entries = 6.98e-09
  b: max relative error over all 4 entries = 1.46e-08
```


**The sampler.** 
Proposition prop:14-gibbsaccept says block Gibbs has the model
distribution as its stationary distribution.  With $M=6$ we can enumerate all
$64$ visible states, compute $p(\bm{x})$ exactly, and compare against the
empirical frequencies of a long chain in total-variation distance:


```
=== does block Gibbs sample the model distribution? ===
     200 sweeps: total-variation distance to the exact p(v) = 0.0103
    2000 sweeps: total-variation distance to the exact p(v) = 0.0034
   20000 sweeps: total-variation distance to the exact p(v) = 0.0008
  (a uniform guess would give 0.3601)
```


The distance falls as the square root of the number of samples,
which is the Monte Carlo rate of Chapter 2 and is visible as
the slope in Figure fig:rbmtraining(b).

### How biased is CD-\texorpdfstring{$k$
{k}?}

Now the question the notebox raised.  We take a machine small enough for the
exact gradient, average the CD-$k$ estimate over sixty independent chains so
that sampling noise is negligible, and compare.


```
=== CD-k is a BIASED estimator of the gradient ===
  k     cos(CD-k, exact)   ||CD-k - exact|| / ||exact||
    1     0.986291          0.3283
    2     0.998131          0.1037
    5     0.999982          0.0095
   10     0.999956          0.0100
   25     0.999963          0.0089
  100     0.999988          0.0068
```


The two columns tell different halves of the story and both matter.  CD-1 is
wrong by *thirty-three per cent* in magnitude -- a bias no amount of
averaging removes, since every one of the sixty chains carries it.  But its
cosine with the true gradient is $0.986$, an angle of about nine degrees.  It
points very nearly the right way and is merely the wrong length.

That is exactly why CD-1 works.  Gradient ascent does not need the gradient; it
needs a direction with positive inner product with the gradient, and a step size
it can tune anyway.  A systematically short vector at nine degrees to the truth
is a perfectly serviceable ascent direction.  By $k=5$ the error is under one
per cent and the remaining discrepancy is the noise floor of our sixty-chain
average.

**What the bias costs.** 
Bias in the gradient need not mean a worse model, so we measure that too.  On
*bars and stripes* -- a standard small benchmark whose $14$ patterns on a
$3\times3$ grid live among $2^{9}=512$ configurations, so the likelihood is
exact -- we train identical machines with CD-1, CD-10 and the exact gradient:


```
bars and stripes 3x3: 14 distinct patterns of 512 possible
log-likelihood of the perfect model: -2.6391 per image

  training signal    final log-likelihood (3 seeds)
  CD-1               -4.3869   (-4.5322 to -4.3061)
  CD-10              -4.2605   (-4.4092 to -4.1716)
  exact gradient     -4.2457   (-4.4597 to -4.1265)
```


CD-10 and the exact gradient are indistinguishable, $-4.261$ against $-4.246$,
a gap of $0.015$ nats that is well inside the seed-to-seed spread.  CD-1 is
measurably but not catastrophically worse at $-4.387$, about $0.14$ nats behind.
So the thirty-three per cent gradient bias of CD-1 costs roughly a tenth of a
nat of likelihood, and ten Gibbs sweeps buy essentially all of it back.  None of
the three reaches the perfect $-2.639$, which is a statement about the capacity
of eight hidden units and three thousand updates rather than about the training
signal.

![a The bias of contrastive divergence against the number of Gibbs sweep](../BookML/BookFigures/chapter14_boltzmann/rbm_training.png)

*(a) The bias of contrastive divergence against the number of Gibbs sweeps, measured against the exact gradient of Theorem thm:14-gradient with the sampling noise averaged away.  The relative error falls from $0.33$ at $k=1$ to under $0.01$ by $k=5$, while the direction is already almost right at $k=1$.  (b) Total-variation distance between the empirical distribution of a block-Gibbs chain and the exact model distribution, against the number of samples drawn; the dashed line is the $n^{-1/2}$ Monte Carlo rate and the dotted line is what a uniform guess would give.  (c) Log-likelihood during training on bars and stripes for the three training signals, with the likelihood of a perfect model marked.*


## Implementations in the libraries

Neither PyTorch nor TensorFlow ships a Boltzmann machine, which is itself
informative: the architecture predates automatic differentiation and its
gradient, Eq. (14.20), is not obtained by backpropagation
through a loss.  It is a difference of two expectations, and one of them comes
from a sampler.  What the libraries provide is the tensor arithmetic and the
optimisers, and the RBM is written on top.


In [ ]:
import torch
import torch.nn as nn


class RBM(nn.Module):
    """Binary-binary RBM, Eq. (14.energyBB).  Note there is no forward()
    returning a loss: the gradient is Eq. (14.rbmgradient), assigned by hand."""

    def __init__(self, M=784, N=256, k=1):
        super().__init__()
        self.W = nn.Parameter(torch.randn(M, N) * 0.01)
        self.a = nn.Parameter(torch.zeros(M))
        self.b = nn.Parameter(torch.zeros(N))
        self.k = k

    def free_energy(self, x):                       # Eq. (14.freeBB)
        return -(x @ self.a) - torch.nn.functional.softplus(
            x @ self.W + self.b).sum(1)

    def gibbs(self, x):                             # Eq. (14.blockgibbs)
        ph = torch.sigmoid(x @ self.W + self.b)
        h = torch.bernoulli(ph)
        px = torch.sigmoid(h @ self.W.t() + self.a)
        return torch.bernoulli(px)

    def cd_loss(self, x):
        """A surrogate whose gradient equals Eq. (14.cdk).

        The negative sample is detached so that no gradient flows through the
        sampler: the chain supplies configurations, not a differentiable path.
        """
        v = x
        for _ in range(self.k):
            v = self.gibbs(v)
        return self.free_energy(x).mean() - self.free_energy(v.detach()).mean()


model = RBM(784, 256, k=1)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(20):
    for xb, _ in train_loader:
        xb = torch.bernoulli(xb.view(-1, 784))      # binarise the pixels
        loss = model.cd_loss(xb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()


The trick in \verb!cd_loss! is worth understanding rather than copying.  The
difference of free energies is *not* the log-likelihood -- that would need
$Z$ -- but its gradient is exactly Eq. (14.23), because
$\nabla F$ at the data gives the positive phase and $\nabla F$ at the samples
gives the estimated negative phase.  The \verb!detach()! is essential: without
it, autograd would try to differentiate through the Bernoulli draws, which is
both meaningless and wrong.

The same in TensorFlow, where the two phases are written out explicitly rather
than routed through a surrogate loss:


In [ ]:
import tensorflow as tf


class RBM(tf.Module):
    def __init__(self, M=784, N=256, k=1):
        self.W = tf.Variable(tf.random.normal([M, N], stddev=0.01))
        self.a = tf.Variable(tf.zeros([M]))
        self.b = tf.Variable(tf.zeros([N]))
        self.k = k

    def sample(self, p):
        return tf.cast(tf.random.uniform(tf.shape(p)) < p, tf.float32)

    def gibbs(self, x):
        h = self.sample(tf.sigmoid(x @ self.W + self.b))
        return self.sample(tf.sigmoid(h @ tf.transpose(self.W) + self.a))

    @tf.function
    def cd_update(self, x, lr=0.01):
        """Eq. (14.rbmgradient) with the negative phase from k Gibbs sweeps."""
        ph_data = tf.sigmoid(x @ self.W + self.b)
        v = x
        for _ in range(self.k):
            v = self.gibbs(v)
        ph_model = tf.sigmoid(v @ self.W + self.b)
        n = tf.cast(tf.shape(x)[0], tf.float32)
        self.W.assign_add(lr * (tf.transpose(x) @ ph_data
                                - tf.transpose(v) @ ph_model) / n)
        self.a.assign_add(lr * tf.reduce_mean(x - v, axis=0))
        self.b.assign_add(lr * tf.reduce_mean(ph_data - ph_model, axis=0))


## Summary and the programs

A Boltzmann machine models the data distribution itself as a canonical ensemble,
Eq. (14.1).  Everything follows from one obstruction: the
partition function (14.2) is a sum over $2^{M+N}$ configurations and is
permanently unavailable.

Theorem thm:14-gradient locates the damage precisely.  The gradient of the
log-likelihood is a difference of two expectations -- one over the data, which is
free, and one over the model, which requires samples.  The value of $Z$ is never
needed, only samples from the distribution it normalises, and that loophole is
what Metropolis (Proposition prop:14-metropolis) and Gibbs
(Proposition prop:14-gibbsaccept) exploit, both by way of detailed
balance.

Theorem thm:14-factorise is what makes the restricted machine practical.
Deleting the within-layer couplings makes both conditionals factorise into
logistic units, so the positive phase becomes exact, Gibbs sampling updates a
whole layer at a time, and the free energy has the closed form
(14.18).  An RBM is a logistic layer read as a probability model.

We verified what could be verified exactly.  The gradient identity matched
finite differences to $10^{-8}$; block Gibbs converged to the enumerated model
distribution at the $n^{-1/2}$ Monte Carlo rate, reaching a total-variation
distance of $0.0008$ against $0.36$ for a uniform guess.

The measurement to remember is Section *How biased is CD-\texorpdfstring{$k$}{k}?*.  CD-1 is wrong by
$33\%$ in magnitude, a bias that averaging cannot remove, and yet its cosine
with the true gradient is $0.986$ -- about nine degrees.  It is a short vector
pointing almost the right way, which is all gradient ascent requires.  On bars
and stripes that bias cost $0.14$ nats of likelihood against the exact gradient,
and ten Gibbs sweeps recovered all but $0.015$ of it.  Approximate gradients are
acceptable when their direction is right; it is worth knowing which of the two
one is relying on.

The programs are in the directory  

`doc/BookML/BookPrograms/chapter14_boltzmann`.  

Every listing above appears there as a numbered file, and three modules run
start to finish and reproduce the numbers quoted in the text:

- `rbm.py` -- the energy and free energy, the exact partition
   function by enumeration, the conditionals, block Gibbs, the exact
   gradient and CD-$k$ with an optional persistent chain.
- `verify_rbm.py` -- the gradient check against finite
   differences, the Gibbs convergence study, and the CD-$k$ bias table.
- `run_bas.py` -- the bars-and-stripes experiment.

The figure is generated by `ch14_figures.py` in
`doc/BookML/BookFigures`; it is not drawn by hand.


## Exercises

### Warm-up exercises

1. **Counting.**
   (a) How many terms has the partition function of an RBM with $M=784$ and
   $N=500$?  Express the answer as a power of ten.
   (b) At $10^{9}$ terms per second, how long would the sum take?
   (c) For which $M$ is enumeration feasible on a laptop, and how does that
   compare with the smallest interesting data set?
2. **The gradient identity.**
   Derive Eq. (14.20) from Theorem thm:14-gradient by
   computing $\partial E/\partial w_{ij}$, $\partial E/\partial a_i$ and
   $\partial E/\partial b_j$ for the energy (14.15).  Then explain
   in one sentence why the positive phase needs no sampling but the negative
   phase does.
3. **Factorisation.**
   Prove Theorem thm:14-factorise for the visible conditional by repeating
   the argument with the roles of $\bm{x}$ and $\bm{h}$ exchanged.  Then show
   that adding a single visible-visible coupling $J_{12}x_1x_2$ destroys the
   factorisation, and say what that costs computationally.
4. **Detailed balance.**
   (a) Verify Proposition prop:14-detailed on a two-state chain.
   (b) Construct a three-state chain that is stationary for the uniform
   distribution but violates detailed balance, confirming that the converse
   fails.
   (c) Show that the Metropolis acceptance (14.12) is unchanged
   if $\pi$ is multiplied by any constant, and explain why this is the fact the
   whole chapter depends on.
5. **Gibbs from Metropolis.**
   Work through Proposition prop:14-gibbsaccept for a two-variable
   distribution of your choice, and confirm numerically that the acceptance
   probability is one for every proposal.  Then find a distribution for which
   Gibbs mixes very slowly, and explain the geometry that causes it.
6. **Free energy.**
   Verify Eq. (14.18) numerically by computing
   $-\log\sum_{\bm{h}}e^{-E}$ by brute force for a machine with $N=10$ and
   comparing.  Then show that $F$ is the log-sum-exp of $N$ terms and explain why
   `logaddexp` rather than `log(1+exp())` is used in the code.

### Project-style exercise: a Boltzmann machine from scratch

**Part a: the machinery.** 
Implement the energy, free energy, exact partition function, both conditionals
and block Gibbs.  Verify Eq. (14.18) against brute-force summation
over $\bm{h}$, and verify Theorem thm:14-gradient by comparing the exact
gradient against central differences of the exact log-likelihood.

**Part b: the sampler.** 
Reproduce the Gibbs convergence study of Section *Implementation and verification*.  Measure the
integrated autocorrelation time of the chain, using the machinery of
Chapter 2, and study how it grows as the weights are scaled
up.  At what coupling strength does the chain stop mixing usefully, and what is
happening physically at that point?

**Part c: the bias.** 
Reproduce the CD-$k$ table of Section *How biased is CD-\texorpdfstring{$k$}{k}?*, separating bias from
variance: average many chains at fixed parameters to isolate the bias, and
report the variance separately.  Then implement persistent contrastive
divergence and place it on the same axes.  Does PCD reduce the bias, the
variance, or both?

**Part d: what the bias costs.** 
Reproduce the bars-and-stripes comparison, then extend it: plot final
log-likelihood against $k$ for $k=1,\dots,20$, and against the number of hidden
units.  Is there a regime in which CD-1 is not merely worse but qualitatively
wrong -- for instance, assigning high probability to configurations the data
never contain?

**Part e: a physical system.** 
Train an RBM on configurations of the two-dimensional Ising model sampled at
several temperatures.  Inspect the learned weights $\bm{W}$: do the hidden units
discover local structure, and does anything change across the critical
temperature?  Compare the model's energy histogram against the true one.  This
is a case where you know the answer, which makes it the right place to find out
whether the method works.

**Part f: Gaussian-binary.** 
Implement the machine of Eq. (14.21) and fit it to continuous
data.  Verify the conditional (14.22) numerically.  Then attempt to
learn the $\sigma_i$ jointly with the weights and document the instability
described in Section *Gaussian-binary machines*; propose and test a remedy.

**Part g: against an autoencoder.** 
Train an RBM and the autoencoder of Chapter 12 on the same data with
the same number of hidden units, and compare what they learn.  The autoencoder
optimises reconstruction and the RBM optimises likelihood: exhibit a case where
the two disagree, and say which you would trust for which purpose.
